In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

In [3]:
from google.colab import files
uploaded=files.upload()

Saving SaaS_Financial_Churn_Dataset.csv to SaaS_Financial_Churn_Dataset.csv


In [4]:
df = pd.read_csv("SaaS_Financial_Churn_Dataset.csv")

In [5]:
print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (150000, 20)
   Customer_ID       Industry Company_Size  Tenure_Months  Monthly_Charges  \
0  CUST_000001         Retail      Startup             27          3242.22   
1  CUST_000002     Healthcare          SMB             46          3411.10   
2  CUST_000003     Technology   Enterprise             43          4671.74   
3  CUST_000004  Manufacturing   Enterprise             45          2996.55   
4  CUST_000005        Finance   Enterprise             30           305.92   

   Total_Users  Integration_Count  Monthly_Logins  API_Utilization_Rate  \
0          419                  3             224                 73.26   
1          260                  7              92                 25.97   
2          368                  2             105                 15.27   
3          197                  6              82                 72.92   
4           68                  7             244                 46.48   

   Premium_Feature_Usage  Support_Tickets_Raised  Sy

In [6]:
df = df.drop("Customer_ID", axis=1)

In [7]:
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(exclude=np.number).columns

In [8]:
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

In [9]:
df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [10]:
le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

In [11]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Size:", X_train.shape)
print("Testing Size:", X_test.shape)

Training Size: (120000, 18)
Testing Size: (30000, 18)


In [13]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("\nRandom Forest Accuracy:", round(rf_acc*100,2), "%")


Random Forest Accuracy: 75.55 %


In [14]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgbm.fit(X_train, y_train)

lgbm_pred = lgbm.predict(X_test)

lgbm_acc = accuracy_score(y_test, lgbm_pred)

print("\nLightGBM Accuracy:", round(lgbm_acc*100,2), "%")


[LightGBM] [Info] Number of positive: 33296, number of negative: 86704
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015881 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1464
[LightGBM] [Info] Number of data points in the train set: 120000, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.277467 -> initscore=-0.957063
[LightGBM] [Info] Start training from score -0.957063

LightGBM Accuracy: 75.61 %


In [15]:
if lgbm_acc > rf_acc:
    print("\nBest Model: LightGBM")
else:
    print("\nBest Model: Random Forest")

print("\nClassification Report")
print(classification_report(y_test, lgbm_pred))


Best Model: LightGBM

Classification Report
              precision    recall  f1-score   support

         0.0       0.78      0.93      0.85     21676
         1.0       0.62      0.31      0.42      8324

    accuracy                           0.76     30000
   macro avg       0.70      0.62      0.63     30000
weighted avg       0.73      0.76      0.73     30000

